# boltU data prep
Tokenizes the configured dataset entirely on Kaggle (no GPU needed) and publishes the shards
as a private Kaggle Dataset for the training kernel to attach. Run with **Accelerator: None,
Internet: On** — tokenization is CPU/network-bound, running it here saves your GPU-hour quota
for actual training. Needs two Kaggle Secrets: `GH_TOKEN` and `KAGGLE_API_TOKEN`.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
gh_tok = secrets.get_secret("GH_TOKEN")
os.environ["KAGGLE_API_TOKEN"] = secrets.get_secret("KAGGLE_API_TOKEN")

if not os.path.isdir("/kaggle/working/boltU"):
    subprocess.run(["git", "clone", f"https://{gh_tok}@github.com/dhmizu95/boltU.git",
                    "/kaggle/working/boltU"], check=True)
os.chdir("/kaggle/working/boltU")
subprocess.run(["git", "pull", "--ff-only"], check=True)

subprocess.run("pip install -q tiktoken pyyaml datasets tqdm kaggle".split(), check=True)

In [ ]:
import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
assert free_gb > 12, f"only {free_gb:.1f} GB free in /kaggle/working"
print(f"{free_gb:.1f} GB free in /kaggle/working")

In [ ]:
subprocess.run(["python", "src/data_prep.py", "--config", "configs/base.yaml"], check=True)

In [ ]:
# Publish data/ as a Kaggle Dataset: init the manifest on first run, version it on re-runs.
import json

DATASET_ID = "delowerhossainmizu/boltu-tokens"
meta_path = "data/dataset-metadata.json"

if not os.path.exists(meta_path):
    subprocess.run(["kaggle", "datasets", "init", "-p", "data"], check=True)
    meta = json.load(open(meta_path))
    meta["id"] = DATASET_ID
    meta["title"] = "boltu-tokens"
    json.dump(meta, open(meta_path, "w"))
    subprocess.run(["kaggle", "datasets", "create", "-p", "data", "--dir-mode", "zip"], check=True)
else:
    subprocess.run(["kaggle", "datasets", "version", "-p", "data", "-m", "retokenized",
                     "--dir-mode", "zip"], check=True)

print(f"published: {DATASET_ID} — attach it in kernel-metadata.json's dataset_sources")